# Plotting the Search

This notebook walks through the plotting helpers in `sklearn-genetic-opt`. It shows the default fitness view, richer history and logbook views, and two search-space plot styles that make it easier to understand how the optimizer explored the parameter space.


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor

from sklearn_genetic import GASearchCV
from sklearn_genetic.plots import plot_fitness_evolution, plot_history, plot_search_space
from sklearn_genetic.space import Categorical, Continuous, Integer

X, y = load_diabetes(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

search = GASearchCV(
    DecisionTreeRegressor(random_state=42),
    cv=2,
    scoring='r2',
    population_size=4,
    generations=5,
    tournament_size=3,
    elitism=True,
    crossover_probability=0.9,
    mutation_probability=0.05,
    param_grid={
        'ccp_alpha': Continuous(0, 1),
        'criterion': Categorical(['squared_error', 'absolute_error']),
        'max_depth': Integer(2, 20),
        'min_samples_split': Integer(2, 30),
    },
    criteria='max',
    n_jobs=1,
)

search.fit(X_train, y_train)

## History and telemetry

The fitted estimator stores generation-level telemetry in `history`. The newer fields track the adaptive behavior of the optimizer, including mutation pressure, diversity control, duplicate handling, and local refinement.


In [ ]:
history = pd.DataFrame(search.history)
telemetry_columns = [
    'gen',
    'fitness_best',
    'fitness',
    'fitness_max',
    'unique_individual_ratio',
    'genotype_diversity',
    'mutation_probability',
    'selection_pressure',
    'random_immigrants',
    'duplicate_replacements',
    'local_refinements',
]
history[[column for column in telemetry_columns if column in history.columns]].tail()

## Fitness plots

`plot_fitness_evolution` now accepts multiple metrics and an optional rolling window, which makes it easier to compare best-so-far fitness with the current population.


In [ ]:
plot_fitness_evolution(search)
plt.show()

In [ ]:
plot_fitness_evolution(
    search,
    metrics=['fitness_best', 'fitness', 'fitness_max'],
    window=2,
    kind='line',
    title='Fitness comparison with smoothing',
)
plt.show()

## History plots

`plot_history` can plot any fields from `history` or `logbook`. Use it to inspect fitness signals, diversity indicators, or optimizer-control events.


In [ ]:
plot_history(
    search,
    fields=['fitness_best', 'fitness', 'unique_individual_ratio', 'genotype_diversity'],
    kind='line',
    subplots=True,
    title='Optimizer history overview',
)
plt.show()

In [ ]:
plot_history(
    search,
    fields=['score', 'fit_time', 'score_time'],
    source='logbook',
    kind='area',
    title='Logbook fields from candidate evaluations',
)
plt.show()

## Search-space plots

The search-space plots now have clearer modes. The pair plot shows relationships between sampled parameters, while the heatmap gives a quick correlation view.


In [ ]:
plot_search_space(
    search,
    features=['ccp_alpha', 'max_depth', 'min_samples_split', 'criterion'],
    hue='criterion',
    kind='pair',
)
plt.show()

In [ ]:
plot_search_space(
    search,
    features=['ccp_alpha', 'max_depth', 'min_samples_split', 'fitness_best'],
    kind='heatmap',
)
plt.show()

## Takeaways

- Use `plot_fitness_evolution` when you want a compact fitness trend.
- Use `plot_history` when you want to inspect telemetry signals like diversity, stagnation, and control events.
- Use `plot_search_space` when you want to understand how the sampled parameter values relate to each other or to the final score.
